# Cartographie thermique 3D — Top-3 modèles : Principal vs Séparé (stationnaire/transitoire)

Ce notebook reprend le pipeline de nettoyage / features du notebook thermique corrigé (sections 0 à 12, identiques), puis compare **3 modèles** (ExtraTrees, RandomForest, XGBoost) selon **2 approches** :
- **Modèle principal** : un seul modèle entraîné sur toutes les données (split temporel).
- **Modèle séparé** : un modèle entraîné séparément sur la phase transitoire et sur la phase stationnaire, puis les prédictions des deux phases sont regroupées pour donner un score global comparable au modèle principal.

**Résultat final** : un tableau de **6 cas** (3 modèles × 2 approches).

## 0 — Installation et imports

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn plotly scipy
print(' Librairies installées!')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings, re, os

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
print('Imports OK')

## 1 — Paramètres de la grille

In [ ]:
# PARAMÈTRES DE LA GRILLE
LARGEUR  = 8   # axe X : A–H  (0–7)
LONGUEUR = 5   # axe Y : 0–4
HAUTEUR  = 3   # niveaux H1–H3  (axe Z : 0–2)
N_CAPTEURS = 120  # nombre de capteurs à utiliser

# Limites physiques des températures (capteurs HS en dehors)
T_MIN_PHYSIQUE = -50
T_MAX_PHYSIQUE = 200

# Espacement vertical entre niveaux (mètres)
ESPACEMENT_Z = 0.5

# POSITIONS DES ÉQUIPEMENTS CVC (à ajuster selon votre plan)
# Format : (x, y) dans la grille  — x=0..7 (A–H), y=0..4
# MODIFIEZ CES VALEURS selon votre document Thermo-couple_distribution !
SOURCES_FROID = [   # Bouches de climatisation
    (0, 0),          # Coin A0 — exemple
    (7, 4),          # Coin H4 — exemple
]
SOURCES_CHAUD = [   # Bouches de chauffage
    (3, 2),          # Centre — exemple
    (4, 2),          # Centre — exemple
]

print(f' Grille configurée : {LARGEUR} × {LONGUEUR} × {HAUTEUR}')
print(f'   Capteurs utilisés : {N_CAPTEURS}')
print(f'   Limites physiques : {T_MIN_PHYSIQUE}°C … {T_MAX_PHYSIQUE}°C')
print(f'   Sources froid (clim) : {SOURCES_FROID}')
print(f'   Sources chaud        : {SOURCES_CHAUD}')

## 2 — Chargement du fichier CSV

In [ ]:
from google.colab import files

print('Sélectionnez votre fichier CSV...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'Fichier : {filename}')

## 3 — Lecture robuste du CSV

In [ ]:
#  LECTURE ROBUSTE DU CSVStories
# Essayer les 4 combinaisons sep/decimal les plus courantes
df_raw = None
for sep, dec in [(',', ','), (';', '.'), (',', '.'), (';', ',')]:
    try:
        tmp = pd.read_csv(filename, sep=sep, decimal=dec, dtype=str)
        if tmp.shape[1] > 10:
            df_raw = tmp
            print(f' CSV lu  sep="{sep}"  decimal="{dec}"')
            break
    except Exception:
        pass

if df_raw is None:
    raise ValueError(' Impossible de lire le CSV — vérifiez le format.')

# ignorer la 1ère ligne si elle est corrompue
# La 1ère ligne de certains exports contient un timestamp en col 0
# et des valeurs de température en col 1 (décalage d'une colonne)
row0 = str(df_raw.iloc[0, 0])
if re.match(r'^\d+:\d+', row0) or re.match(r'^\d{2}:\d{2}\.\d+', row0):
    print(f'  Ligne 0 corrompue détectée (valeur={row0!r}) → ignorée')
    df_raw = df_raw.iloc[1:].reset_index(drop=True)

print(f'Shape brut : {df_raw.shape}')
print(f'Colonnes (6 premières) : {list(df_raw.columns[:6])}')
print('\n3 premières lignes (colonnes 0–5) :')
print(df_raw.iloc[:3, :6].to_string())

## 4 — Nettoyage des températures

In [ ]:
df = df_raw.copy()

# 1. Identifier les colonnes de température
temp_cols_all = [c for c in df.columns
                 if re.match(r'Temperature\d+', c, re.I) or
                    re.match(r'Temp\d+', c, re.I)]
temp_cols = temp_cols_all[:N_CAPTEURS]
print(f' {len(temp_cols_all)} colonnes température → {len(temp_cols)} utilisées')

# 2. Convertir en float (virgule ou point décimal)
for col in temp_cols:
    df[col] = (
        df[col].astype(str)
               .str.replace(',', '.', regex=False)
               .pipe(pd.to_numeric, errors='coerce')
    )

row_col  = df.columns[0]
time_col = df.columns[1]

# 3. Supprimer valeurs physiquement impossibles
n_bad = 0
for col in temp_cols:
    mask = (df[col] < T_MIN_PHYSIQUE) | (df[col] > T_MAX_PHYSIQUE)
    n_bad += mask.sum()
    df.loc[mask, col] = np.nan
print(f' {n_bad} valeurs hors limites physiques neutralisées (NaN)')

# 4. Clipping 1%–99%
for col in temp_cols:
    p1, p99 = df[col].quantile([0.01, 0.99])
    df[col] = df[col].clip(p1, p99)
print(' Clipping 1%–99% effectué')

# 5. Température moyenne par ligne (pour référence uniquement)
df['temperature_mean'] = df[temp_cols].mean(axis=1, skipna=True)

n_avant = len(df)
df = df.dropna(subset=['temperature_mean'])
print(f' Lignes conservées : {n_avant} → {len(df)}')

# 6. Statistiques individuelles par capteur
sensor_stats = df[temp_cols].describe().T
print(f'\n Variation spatiale (std par capteur) :')
print(f'   Min std : {sensor_stats["std"].min():.3f}°C')
print(f'   Max std : {sensor_stats["std"].max():.3f}°C')
print(f'   Moy std : {sensor_stats["std"].mean():.3f}°C')
print(f'\n Variation spatiale instantanée (std inter-capteurs par ligne) :')
spatial_std = df[temp_cols].std(axis=1)
print(f'   Moy : {spatial_std.mean():.3f}°C  |  Max : {spatial_std.max():.3f}°C')

## 5 — Visualisation post-nettoyage

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Distribution températures après nettoyage', fontsize=13, fontweight='bold')

sample_cols = temp_cols[::max(1, len(temp_cols)//4)][:4]
df[sample_cols].plot(kind='box', ax=axes[0])
axes[0].set_title('Boîtes à moustaches (4 capteurs)')
axes[0].set_ylabel('°C'); axes[0].grid(True, alpha=0.3)

vals = df[temp_cols].values.flatten()
vals = vals[~np.isnan(vals)]
axes[1].hist(vals, bins=50, color='coral', edgecolor='black', alpha=0.8)
axes[1].set_title('Distribution globale'); axes[1].set_xlabel('°C'); axes[1].grid(True, alpha=0.3)

axes[2].plot(df[temp_cols].mean().values, 'o-', ms=3, lw=1.2, color='steelblue')
axes[2].set_title('Moyenne par capteur'); axes[2].set_xlabel('N° capteur'); axes[2].set_ylabel('°C'); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print('Nettoyage terminé')

## 6 — Parsing du temps (`t_sec`)

In [ ]:
# PARSING DU TEMPS
print(f'Colonne temps : "{time_col}"')
print(f'   Exemples : {df[time_col].head(3).tolist()}')

time_ok = False
for fmt in ['%I:%M:%S %p', '%H:%M:%S', '%M:%S', '%H:%M:%S.%f', '%H:%M']:
    try:
        parsed = pd.to_datetime(df[time_col].astype(str).str.strip(), format=fmt, errors='coerce')
        if parsed.notna().sum() >= max(1, len(df) * 0.5):
            df = df.copy()
            df['time'] = parsed
            t0 = df['time'].min()
            df['t_sec'] = (df['time'] - t0).dt.total_seconds()
            print(f' Temps parsé (format {fmt}) : {df["t_sec"].min():.0f}s – {df["t_sec"].max():.0f}s')
            time_ok = True
            break
    except Exception:
        pass

if not time_ok:
    print(' Parsing temps échoué → index × 20s utilisé')
    df = df.copy()
    df['t_sec'] = np.arange(len(df)) * 20.0

## 7 — Diagnostic stationnaire / transitoire

Avant d'aller plus loin, on vérifie si le système a atteint l'équilibre thermique.
Un régime transitoire (température qui dérive encore) pollue les features spatiales
du modèle si on l'ignore.

**Étape 1** : test global (régression température ~ temps, sur tout le run).
**Étape 2** : test par fenêtres glissantes (où se situe la transition, si elle existe).
**Étape 3** : détection automatique du point de bascule et scission du dataset.
**Étape 4** : visualisation des deux régimes.

In [ ]:
from scipy import stats

def test_stationnarite(t, T, seuil_pente_C_par_h=0.5, seuil_p=0.05, verbose=True):
    """Teste si (t, T) est stationnaire (pente non significative), transitoire
    (dérive significative et forte) ou quasi-stationnaire (dérive significative
    mais faible). Retourne un dict avec verdict, pente, r2, p_value."""
    t = np.asarray(t, dtype=float)
    T = np.asarray(T, dtype=float)
    mask = ~np.isnan(t) & ~np.isnan(T)
    t, T = t[mask], T[mask]
    if len(t) < 5:
        return {'verdict': 'INDÉTERMINÉ', 'pente_C_par_h': np.nan, 'r2': np.nan, 'p_value': np.nan}

    pente, ordonnee, r_value, p_value, std_err = stats.linregress(t, T)
    pente_C_par_h = pente * 3600
    ic95 = 1.96 * std_err * 3600

    if p_value > seuil_p:
        verdict = 'STATIONNAIRE'
    elif abs(pente_C_par_h) <= seuil_pente_C_par_h:
        verdict = 'QUASI-STATIONNAIRE'
    else:
        verdict = 'TRANSITOIRE'

    if verbose:
        print(f'{"="*55}')
        print('TEST DE STATIONNARITÉ THERMIQUE')
        print(f'{"="*55}')
        print(f'Durée totale de mesure : {t.max()/60:.1f} min ({t.max()/3600:.2f} h)')
        print(f'Pente                  : {pente_C_par_h:+.4f} °C/h  (± {ic95:.4f})')
        print(f'R² (régression)        : {r_value**2:.4f}')
        print(f'p-value                : {p_value:.4g}')
        print(f'{"="*55}')
        print(f'>>> VERDICT : {verdict}\n')

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.scatter(t / 60, T, s=15, alpha=0.5, color='steelblue', label='Mesures')
        t_fit = np.linspace(t.min(), t.max(), 100)
        ax.plot(t_fit / 60, pente * t_fit + ordonnee, 'r-', lw=2,
                label=f'Tendance : {pente_C_par_h:+.3f} °C/h')
        ax.set_xlabel('Temps (min)'); ax.set_ylabel('Température (°C)')
        ax.set_title(f'{verdict} — R²={r_value**2:.3f}, p={p_value:.3g}')
        ax.legend(); ax.grid(True, alpha=0.3)
        plt.tight_layout(); plt.show()

    return {'verdict': verdict, 'pente_C_par_h': pente_C_par_h, 'r2': r_value**2, 'p_value': p_value}


print('ÉTAPE 1 — Test global')
resultat_global = test_stationnarite(
    df['t_sec'].values, df['temperature_mean'].values
)


In [ ]:
def detecter_phases(df, time_col='t_sec', temp_col='temperature_mean',
                     fenetre_min=30, pas_min=5,
                     seuil_pente_C_par_h=0.5, seuil_p=0.05):
    """Fait glisser une fenêtre temporelle et teste la stationnarité locale
    dans chaque fenêtre. Retourne un DataFrame : t_debut_min, t_fin_min,
    pente_C_par_h, r2, p_value, verdict."""
    t_all = df[time_col].values.astype(float)
    T_all = df[temp_col].values.astype(float)

    fenetre_s = fenetre_min * 60
    pas_s     = pas_min * 60
    t_min, t_max = np.nanmin(t_all), np.nanmax(t_all)

    records = []
    t_debut = t_min
    while t_debut + fenetre_s <= t_max:
        t_fin = t_debut + fenetre_s
        m = (t_all >= t_debut) & (t_all < t_fin)
        if m.sum() >= 10:
            res = test_stationnarite(t_all[m], T_all[m],
                                      seuil_pente_C_par_h, seuil_p, verbose=False)
            records.append({'t_debut_min': t_debut / 60, 't_fin_min': t_fin / 60, **res})
        t_debut += pas_s

    phases = pd.DataFrame(records)
    print(f'{len(phases)} fenêtres analysées ({fenetre_min} min, pas {pas_min} min)')
    print(phases['verdict'].value_counts().to_string())
    return phases


print('ÉTAPE 2 — Test par fenêtres glissantes (30 min, pas de 5 min)')
phases = detecter_phases(df, fenetre_min=30, pas_min=5)


In [ ]:
def scinder_stationnaire(df, phases, time_col='t_sec', n_fenetres_stables=3):
    """Cherche la première série de 'n_fenetres_stables' fenêtres consécutives
    STATIONNAIRE/QUASI-STATIONNAIRE et scinde df en (transitoire, stationnaire)
    à cette frontière."""
    est_stable = phases['verdict'].isin(['STATIONNAIRE', 'QUASI-STATIONNAIRE'])

    t_bascule_min = None
    for i in range(len(phases) - n_fenetres_stables + 1):
        if est_stable.iloc[i:i + n_fenetres_stables].all():
            t_bascule_min = phases['t_debut_min'].iloc[i]
            break

    if t_bascule_min is None:
        print(' Aucune phase stationnaire stable détectée -> le système ne semble '
              'jamais se stabiliser sur la durée mesurée.')
        return df.copy(), df.iloc[0:0].copy(), None

    t_bascule_s = t_bascule_min * 60
    df_transitoire  = df[df[time_col] <  t_bascule_s].copy()
    df_stationnaire = df[df[time_col] >= t_bascule_s].copy()

    print(f'\n Bascule détectée à t = {t_bascule_min:.1f} min ({t_bascule_min/60:.2f} h)')
    print(f'   Phase TRANSITOIRE  : {len(df_transitoire):,} lignes  (0 -> {t_bascule_min:.1f} min)')
    print(f'   Phase STATIONNAIRE : {len(df_stationnaire):,} lignes  ({t_bascule_min:.1f} min -> fin)')

    return df_transitoire, df_stationnaire, t_bascule_min


print('ÉTAPE 3 — Détection de la bascule et scission du dataset')
df_transitoire, df_stationnaire, t_bascule = scinder_stationnaire(df, phases)


In [ ]:
def visualiser_phases(df, phases, t_bascule_min, time_col='t_sec', temp_col='temperature_mean'):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    t_min_arr = df[time_col].values / 60
    ax1.scatter(t_min_arr, df[temp_col], s=8, alpha=0.4, color='steelblue')
    if t_bascule_min is not None:
        ax1.axvline(t_bascule_min, color='red', ls='--', lw=2,
                    label=f'Bascule à {t_bascule_min:.0f} min')
        ax1.axvspan(0, t_bascule_min, color='orange', alpha=0.1, label='Transitoire')
        ax1.axvspan(t_bascule_min, t_min_arr.max(), color='green', alpha=0.1, label='Stationnaire')
    ax1.set_ylabel('Température (°C)')
    ax1.set_title('Température brute et zones détectées')
    ax1.legend(); ax1.grid(True, alpha=0.3)

    couleurs = {'STATIONNAIRE': 'green', 'QUASI-STATIONNAIRE': 'gold',
                'TRANSITOIRE': 'red', 'INDÉTERMINÉ': 'gray'}
    for _, row in phases.iterrows():
        ax2.barh(0, row['t_fin_min'] - row['t_debut_min'], left=row['t_debut_min'],
                  color=couleurs.get(row['verdict'], 'gray'), edgecolor='white')
    ax2.set_yticks([])
    ax2.set_xlabel('Temps (min)')
    ax2.set_title('Verdict de stationnarité par fenêtre glissante')
    plt.tight_layout(); plt.show()


print('ÉTAPE 4 — Visualisation')
visualiser_phases(df, phases, t_bascule)


### Test ADF (Augmented Dickey-Fuller) — confirmation rigoureuse

La régression linéaire (étape 1) teste une tendance *linéaire*. Le test ADF est
plus général : son hypothèse nulle (H0) est que la série a une **racine unitaire**
(non-stationnaire, dérive de type marche aléatoire, pas forcément linéaire).

- **p-value < 0.05** → on rejette H0 → la série est **stationnaire** (au sens statistique strict).
- **p-value ≥ 0.05** → on ne peut pas rejeter H0 → série **non-stationnaire**.

On l'applique à `temperature_mean` triée par `t_sec`, globalement puis sur chaque
phase détectée à l'étape 3 (si elle existe).

In [ ]:
!pip install -q statsmodels
from statsmodels.tsa.stattools import adfuller

def test_adf(T, nom='série', verbose=True):
    """Applique le test ADF à une série 1D (déjà triée dans le temps).
    Retourne un dict {statistique, p_value, verdict, valeurs_critiques}."""
    T = np.asarray(T, dtype=float)
    T = T[~np.isnan(T)]
    if len(T) < 10:
        return {'verdict': 'INDÉTERMINÉ', 'p_value': np.nan, 'statistique': np.nan}

    stat, p_value, lags, nobs, crit_vals, icbest = adfuller(T, autolag='AIC')
    verdict = 'STATIONNAIRE (ADF)' if p_value < 0.05 else 'NON-STATIONNAIRE (ADF)'

    if verbose:
        print(f'--- ADF : {nom} ---')
        print(f'  n = {nobs}, lags = {lags}')
        print(f'  Statistique ADF = {stat:.4f}')
        print(f'  p-value         = {p_value:.4g}')
        print(f'  Valeurs critiques : ' + ', '.join(f'{k}={v:.3f}' for k, v in crit_vals.items()))
        print(f'  >>> {verdict}\n')

    return {'verdict': verdict, 'p_value': p_value, 'statistique': stat, 'valeurs_critiques': crit_vals}


print('ADF — série complète')
df_tri = df.sort_values('t_sec')
adf_global = test_adf(df_tri['temperature_mean'].values, nom='dataset complet')

if t_bascule is not None:
    print('ADF — phase transitoire')
    adf_transitoire = test_adf(
        df_transitoire.sort_values('t_sec')['temperature_mean'].values, nom='phase transitoire'
    )
    print('ADF — phase stationnaire')
    adf_stationnaire = test_adf(
        df_stationnaire.sort_values('t_sec')['temperature_mean'].values, nom='phase stationnaire'
    )
else:
    print('Pas de bascule détectée -> ADF sur la série complète uniquement (ci-dessus).')


### Option — n'utiliser que la phase stationnaire pour la suite

Si une phase stationnaire a été détectée, tu peux restreindre `df` à cette
phase avant de construire le format long et d'entraîner le modèle : le signal
spatial ne sera alors plus pollué par la dérive temporelle.

Mets `FILTRER_STATIONNAIRE = False` pour garder tout le dataset (comportement
d'origine, avec `t_sec` comme feature pour absorber la dérive).

In [ ]:
FILTRER_STATIONNAIRE = False  # -> True pour n'entraîner que sur la phase stationnaire

if FILTRER_STATIONNAIRE and t_bascule is not None and len(df_stationnaire) > 0:
    print(f'Filtrage activé : df passe de {len(df):,} à {len(df_stationnaire):,} lignes '
          f'(phase stationnaire uniquement, t >= {t_bascule:.1f} min).')
    df = df_stationnaire.reset_index(drop=True)
else:
    print(f'Filtrage désactivé (ou aucune bascule trouvée) : df conservé tel quel '
          f'({len(df):,} lignes). t_sec reste une feature du modèle pour capturer la dérive.')


## 8 — Détection de la direction aller-retour

In [ ]:
#  DÉTECTION DIRECTION ALLER-RETOUR
print('\n Détection direction aller-retour du robot...')

df = df.copy()
if 'Direction' in df.columns or 'direction' in df.columns:
    dir_col = 'Direction' if 'Direction' in df.columns else 'direction'
    df['direction'] = df[dir_col].str.upper()
    print(f' Colonne "{dir_col}" utilisée')
else:
    df['t_diff']   = df['t_sec'].diff()
    df['is_jump']  = df['t_diff'] < 0
    df['cycle_num'] = df['is_jump'].cumsum()
    df['direction'] = df['cycle_num'].apply(lambda x: 'ALLER' if x % 2 == 0 else 'RETOUR')
    print(f' Direction détectée automatiquement — {df["cycle_num"].max() + 1} cycle(s)')

df['direction_encode'] = (df['direction'] == 'RETOUR').astype(int)
print(f'   ALLER:  {(df["direction_encode"]==0).sum()} mesures')
print(f'   RETOUR: {(df["direction_encode"]==1).sum()} mesures')

## 9 — Mapping capteur → position (x, y, z)

In [ ]:
# ── MAPPING CAPTEUR → POSITION (x, y, z)
# Grille 8×5×3 = 120 capteurs
# Capteurs numérotés : d'abord par hauteur (Z), puis par colonne X, puis par rangée Y
# Exemple : capteur 1 = (x=0,y=0,z=0)=A0H1, capteur 2 = (x=0,y=0,z=1)=A0H2, etc.
#
# ADAPTEZ ce mapping selon votre document Thermo-couple_distribution !
# Si votre document donne un ordre différent, modifiez la logique ci-dessous.

def build_sensor_positions(largeur, longueur, hauteur):
    """
    Construit le dictionnaire {numéro_capteur: (x, y, z)}
    Ordre : Z varie en premier (H1→H3), puis X (A→H), puis Y (0→4)
    → capteur 1=(0,0,0), 2=(0,0,1), 3=(0,0,2), 4=(0,1,0), ...
    """
    pos = {}
    n = 1
    for y in range(longueur):       # rangées 0–4
        for x in range(largeur):    # colonnes A–H (0–7)
            for z in range(hauteur): # hauteurs H1–H3 (0–2)
                pos[n] = (x, y, z)
                n += 1
    return pos

capteur_positions = build_sensor_positions(LARGEUR, LONGUEUR, HAUTEUR)
print(f' {len(capteur_positions)} positions capteurs construites')
print('   Exemples :')
for k in [1, 2, 3, 4, LARGEUR*HAUTEUR, len(capteur_positions)]:
    if k in capteur_positions:
        x, y, z = capteur_positions[k]
        label = f'{chr(ord("A")+x)}{y}H{z+1}'
        print(f'   Capteur {k:3d} → {label} (x={x}, y={y}, z={z})')

## 10 — Construction du DataFrame long

In [ ]:
# CONSTRUCTION DU DATAFRAME LONG
# 1 ligne = 1 capteur × 1 instant  →  N_lignes_CSV × N_CAPTEURS lignes au total

print(' Construction du format long...')
records = []

for _, row in df.iterrows():
    t_val        = float(row['t_sec'])
    dir_val      = int(row['direction_encode'])

    for i, col in enumerate(temp_cols):
        sensor_num = i + 1           # numérotation 1-based
        temp_val   = row[col]        # température individuelle du capteur

        if pd.isna(temp_val):
            continue                 # ignorer les NaN

        pos = capteur_positions.get(sensor_num)
        if pos is None:
            continue

        x, y, z = pos
        records.append({
            'sensor_id':        sensor_num,
            'x':                x,
            'y':                y,
            'z':                z,
            't_sec':            t_val,
            'direction_encode': dir_val,
            'temperature':      float(temp_val),  # ← température INDIVIDUELLE
        })

df_long = pd.DataFrame(records)
print(f' Format long créé : {len(df_long):,} lignes  ({len(df)} instants × {len(temp_cols)} capteurs)')
print(f'\n Statistiques température individuelle :')
print(f'   Min : {df_long["temperature"].min():.2f}°C')
print(f'   Max : {df_long["temperature"].max():.2f}°C')
print(f'   Moy : {df_long["temperature"].mean():.2f}°C')
print(f'   Std : {df_long["temperature"].std():.2f}°C  ← variation réelle !')

## 11 — Features CVC (distance aux sources chaud/froid)

In [ ]:
# ── POSITIONS ET TYPE DE DIFFUSION DES ÉQUIPEMENTS CVC ──────────────────
# Chaque source est un dict : position (x, y, z) + type_diffusion.
#   'horizontale' : l'air chaud/froid se propage sur le plan (x,y),
#                   reste concentré à sa hauteur -> décroît vite en z.
#   'verticale'   : l'air chaud/froid monte/descend dans la pièce,
#                   reste présent à toutes les hauteurs -> décroît vite en (x,y).
#
# ADAPTE ici : position (x=0..LARGEUR-1, y=0..LONGUEUR-1, z=0..HAUTEUR-1)
# et type_diffusion pour chaque équipement réel.
SOURCES_FROID = []  # Pas de climatisation
SOURCES_CHAUD = [
    {'x': 7, 'y': 2, 'z': 1, 'type_diffusion': 'horizontale'},  # Heater — Est
    {'x': 7, 'y': 3, 'z': 1, 'type_diffusion': 'horizontale'},  # Heater — Est
]

# Poids anisotropes par type de diffusion — PARAMÉTRABLE : plus un poids est
# grand, plus la distance (donc la perte d'influence) augmente vite dans cette
# direction. Ajuste ces valeurs selon le comportement réel de tes équipements.
POIDS_DIFFUSION = {
    'horizontale': {'xy': 1.0, 'z': 2.5},   # décroît vite en hauteur
    'verticale':   {'xy': 2.5, 'z': 0.3},   # décroît vite latéralement
}

def dist_min_cvc(x, y, z, sources, espacement_z, poids_diffusion=POIDS_DIFFUSION):
    """Distance pondérée minimale vers le groupe de sources le plus proche,
    en tenant compte du type de diffusion (horizontale/verticale) de chaque
    source. Retourne (distance, type_diffusion_de_la_source_la_plus_proche)."""
    if not sources:
        return 99.0, 'aucune'
    candidats = []
    for s in sources:
        sx, sy = s['x'], s['y']
        sz = s.get('z', 0)
        type_d = s.get('type_diffusion', 'horizontale')
        poids = poids_diffusion[type_d]
        d_xy = np.sqrt((x - sx) ** 2 + (y - sy) ** 2)
        d_z = abs(z - sz) * espacement_z
        d = np.sqrt((poids['xy'] * d_xy) ** 2 + (poids['z'] * d_z) ** 2)
        candidats.append((d, type_d))
    return min(candidats, key=lambda t: t[0])

cx = (LARGEUR  - 1) / 2
cy = (LONGUEUR - 1) / 2
cz = (HAUTEUR  - 1) / 2 * ESPACEMENT_Z

df_long = df_long.copy()

# Features géométriques
df_long['dist_edge_x']  = np.minimum(df_long['x'], (LARGEUR  - 1) - df_long['x'])
df_long['dist_edge_y']  = np.minimum(df_long['y'], (LONGUEUR - 1) - df_long['y'])
df_long['dist_edge_z']  = np.minimum(
    df_long['z'] * ESPACEMENT_Z,
    (HAUTEUR - 1) * ESPACEMENT_Z - df_long['z'] * ESPACEMENT_Z
)
df_long['dist_center']  = np.sqrt(
    (df_long['x'] - cx)**2 +
    (df_long['y'] - cy)**2 +
    (df_long['z'] * ESPACEMENT_Z - cz)**2
)

# ── Features CVC (distance pondérée + type de diffusion dominant) ──────
_froid = df_long.apply(
    lambda r: dist_min_cvc(r['x'], r['y'], r['z'], SOURCES_FROID, ESPACEMENT_Z), axis=1
)
df_long['dist_froid'] = _froid.apply(lambda t: t[0])
df_long['froid_diffusion_verticale'] = (_froid.apply(lambda t: t[1]) == 'verticale').astype(int)

_chaud = df_long.apply(
    lambda r: dist_min_cvc(r['x'], r['y'], r['z'], SOURCES_CHAUD, ESPACEMENT_Z), axis=1
)
df_long['dist_chaud'] = _chaud.apply(lambda t: t[0])
df_long['chaud_diffusion_verticale'] = (_chaud.apply(lambda t: t[1]) == 'verticale').astype(int)

# Score d'influence CVC (plus proche = plus influencé)
df_long['influence_froid'] = 1.0 / (1.0 + df_long['dist_froid'])
df_long['influence_chaud'] = 1.0 / (1.0 + df_long['dist_chaud'])

print('✅ Features créées :')
for col in ['dist_edge_x','dist_edge_y','dist_edge_z','dist_center',
            'dist_froid','dist_chaud','influence_froid','influence_chaud',
            'froid_diffusion_verticale','chaud_diffusion_verticale']:
    print(f'   {col:26s}: {df_long[col].min():.2f} – {df_long[col].max():.2f}')

# Visualiser l'influence CVC sur la grille 2D (coupe à hauteur z=0, à titre indicatif :
# avec des sources verticales, la carte serait quasi identique à toutes les hauteurs)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sources, title, cmap in zip(
    axes,
    [SOURCES_FROID, SOURCES_CHAUD],
    ['Influence climatisation (froid)', 'Influence chauffage (chaud)'],
    ['Blues_r', 'Reds']
):
    grid_inf = np.zeros((LARGEUR, LONGUEUR))
    for xi in range(LARGEUR):
        for yi in range(LONGUEUR):
            d, _ = dist_min_cvc(xi, yi, 0, sources, ESPACEMENT_Z)
            grid_inf[xi, yi] = 1.0 / (1.0 + d)
    im = ax.imshow(grid_inf.T, origin='lower', cmap=cmap, aspect='auto')
    ax.set_xticks(range(LARGEUR))
    ax.set_xticklabels([chr(ord('A')+i) for i in range(LARGEUR)])
    ax.set_yticks(range(LONGUEUR))
    ax.set_yticklabels(range(LONGUEUR))
    ax.set_xlabel('X (A–H)'); ax.set_ylabel('Y (0–4)')
    ax.set_title(title)
    for s in sources:
        marker = 'D' if s.get('type_diffusion') == 'verticale' else '*'
        ax.scatter(s['x'], s['y'], s=200, marker=marker, c='gold', edgecolors='black', zorder=5)
    plt.colorbar(im, ax=ax, label='Influence (0–1)')

plt.suptitle('Carte d\'influence CVC sur la grille (★ horizontale, ◆ verticale)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('✅ Carte CVC générée')

# ── Nouvelles features physiques (transitoire thermique) ───────────────
# Tri obligatoire par capteur puis par temps pour calculer dT_dt et le lag
df_long = df_long.sort_values(['sensor_id', 't_sec']).reset_index(drop=True)

# dT/dt : vitesse de variation de température par capteur (inertie thermique)
# -> probablement le gain le plus fort pour capturer le régime transitoire
_dT = df_long.groupby('sensor_id')['temperature'].diff()
_dt = df_long.groupby('sensor_id')['t_sec'].diff()
df_long['dT_dt'] = (_dT / _dt).replace([np.inf, -np.inf], np.nan).fillna(0.0)

# Lag feature : température du même capteur à l'instant précédent
# -> capture l'inertie thermique (état précédent du système)
df_long['temp_lag1'] = df_long.groupby('sensor_id')['temperature'].shift(1)
df_long['temp_lag1'] = df_long['temp_lag1'].fillna(df_long['temperature'])

# Interaction dist_center * t_sec : la convergence vers l'équilibre dépend
# à la fois de la position ET du temps écoulé
df_long['dist_center_x_t'] = df_long['dist_center'] * df_long['t_sec']

print('✅ Features physiques transitoires créées : dT_dt, temp_lag1, dist_center_x_t')
for col in ['dT_dt', 'temp_lag1', 'dist_center_x_t']:
    print(f'   {col:20s}: {df_long[col].min():.3f} – {df_long[col].max():.3f}')


## 12 — Préparation des features (X, y) et normalisation

In [ ]:
feature_cols = [
    'x', 'y', 'z', 't_sec',
    'dist_edge_x', 'dist_edge_y', 'dist_edge_z', 'dist_center',
    'direction_encode',
    'dist_froid', 'dist_chaud',
    'influence_froid', 'influence_chaud',
    'froid_diffusion_verticale', 'chaud_diffusion_verticale',  # ← type CVC (horizontale/verticale)
    'dT_dt', 'temp_lag1', 'dist_center_x_t',
]
target_col = 'temperature'
if len(df_long) == 0:
    raise ValueError(' Dataset vide!')

X = df_long[feature_cols].values
y = df_long[target_col].values

print(f' Shape : X={X.shape}, y={y.shape}')
print(f'   Variation cible : {y.std():.3f}°C  (std — doit être > 0.5°C)')

scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

print('\n Normalisation OK')
print(f'   Moyennes : {X_scaled.mean(axis=0).round(3)}')
print(f'   Std      : {X_scaled.std(axis=0).round(3)}')

## 13 — Modèle principal (Top-3 modèles, split temporel)

Un seul modèle par algorithme, entraîné sur **toutes** les données (sans distinguer
transitoire/stationnaire), évalué sur un split temporel (80% anciens -> train,
20% récents -> test).

In [ ]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

# ── Split temporel (par rang chronologique, pas par valeur) ─────────────
t_sec_values = df_long['t_sec'].values
order = np.argsort(t_sec_values, kind='stable')
n_total = len(t_sec_values)
split_point = int(n_total * 0.8)
split_point = min(max(split_point, 1), n_total - 1)
train_idx, test_idx = order[:split_point], order[split_point:]

X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f'Modèle principal — Train : {len(X_train):,} | Test : {len(X_test):,}')

top3_models = {
    'ExtraTrees':   ExtraTreesRegressor(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1),
    'RandomForest': RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1),
    'XGBoost':      XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05,
                                  subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                                  random_state=42, verbosity=0),
}

resultats_principal = []
predictions_principal = {}   # nom -> (y_test, y_pred), réutilisé section 15

for nom, m in top3_models.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae  = mean_absolute_error(y_test, pred)
    r2   = r2_score(y_test, pred)
    resultats_principal.append({'Modèle': nom, 'Approche': 'Principal',
                                 'RMSE (°C)': rmse, 'MAE (°C)': mae, 'R2': r2,
                                 'n_test': len(y_test)})
    predictions_principal[nom] = (y_test.copy(), pred.copy())
    print(f'{nom:14s} RMSE={rmse:.4f}°C  MAE={mae:.4f}°C  R²={r2:.4f}')

tableau_principal = pd.DataFrame(resultats_principal).round(4)
print()
display(tableau_principal)


## 14 — Modèle séparé par phase (Top-3 modèles × transitoire/stationnaire)

Même pipeline de features (dont `dT_dt`, `temp_lag1`, `dist_center_x_t`), mais
appliqué séparément à `df_transitoire` et `df_stationnaire`, avec un split temporel
propre à chaque phase. Les prédictions des deux phases sont ensuite regroupées par
modèle pour obtenir un score global "Séparé", comparable au modèle principal.

Nécessite qu'une bascule ait été détectée à l'étape 7 (sinon cette section est
sautée automatiquement).

In [ ]:
def assurer_colonnes(subset, df_ref, cols, key='t_sec'):
    """Rajoute à `subset` les colonnes manquantes (ex: direction_encode) en les
    récupérant depuis df_ref, en faisant correspondre les lignes par t_sec."""
    manquantes = [c for c in cols if c not in subset.columns]
    if not manquantes:
        return subset
    ref = df_ref[[key] + manquantes].drop_duplicates(subset=key)
    return subset.merge(ref, on=key, how='left')


def construire_features_phase(df_subset, capteur_positions, temp_cols,
                                sources_froid, sources_chaud,
                                largeur, longueur, hauteur, espacement_z):
    """Reconstruit le format long + toutes les features (géométriques, CVC,
    physiques dT_dt/temp_lag1/interaction) pour un sous-ensemble de df."""
    records = []
    for _, row in df_subset.iterrows():
        t_val   = float(row['t_sec'])
        dir_val = int(row['direction_encode'])
        for i, col in enumerate(temp_cols):
            sensor_num = i + 1
            temp_val = row[col]
            if pd.isna(temp_val):
                continue
            pos = capteur_positions.get(sensor_num)
            if pos is None:
                continue
            x, y_, z = pos
            records.append({
                'sensor_id': sensor_num, 'x': x, 'y': y_, 'z': z,
                't_sec': t_val, 'direction_encode': dir_val,
                'temperature': float(temp_val),
            })
    dfl = pd.DataFrame(records)
    if len(dfl) < 30:
        return dfl

    dfl = dfl.sort_values(['sensor_id', 't_sec']).reset_index(drop=True)

    cx = (largeur - 1) / 2
    cy = (longueur - 1) / 2
    cz = (hauteur - 1) / 2 * espacement_z

    dfl['dist_edge_x'] = np.minimum(dfl['x'], (largeur - 1) - dfl['x'])
    dfl['dist_edge_y'] = np.minimum(dfl['y'], (longueur - 1) - dfl['y'])
    dfl['dist_edge_z'] = np.minimum(dfl['z'] * espacement_z, (hauteur - 1) * espacement_z - dfl['z'] * espacement_z)
    dfl['dist_center'] = np.sqrt((dfl['x'] - cx)**2 + (dfl['y'] - cy)**2 + (dfl['z'] * espacement_z - cz)**2)
    _froid = dfl.apply(lambda r: dist_min_cvc(r['x'], r['y'], r['z'], sources_froid, espacement_z), axis=1)
    dfl['dist_froid'] = _froid.apply(lambda t: t[0])
    dfl['froid_diffusion_verticale'] = (_froid.apply(lambda t: t[1]) == 'verticale').astype(int)

    _chaud = dfl.apply(lambda r: dist_min_cvc(r['x'], r['y'], r['z'], sources_chaud, espacement_z), axis=1)
    dfl['dist_chaud'] = _chaud.apply(lambda t: t[0])
    dfl['chaud_diffusion_verticale'] = (_chaud.apply(lambda t: t[1]) == 'verticale').astype(int)

    dfl['influence_froid'] = 1.0 / (1.0 + dfl['dist_froid'])
    dfl['influence_chaud'] = 1.0 / (1.0 + dfl['dist_chaud'])

    _dT = dfl.groupby('sensor_id')['temperature'].diff()
    _dt = dfl.groupby('sensor_id')['t_sec'].diff()
    dfl['dT_dt'] = (_dT / _dt).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    dfl['temp_lag1'] = dfl.groupby('sensor_id')['temperature'].shift(1)
    dfl['temp_lag1'] = dfl['temp_lag1'].fillna(dfl['temperature'])
    dfl['dist_center_x_t'] = dfl['dist_center'] * dfl['t_sec']

    return dfl


fcols_phase = ['x', 'y', 'z', 't_sec', 'dist_edge_x', 'dist_edge_y', 'dist_edge_z',
               'dist_center', 'direction_encode', 'dist_froid', 'dist_chaud',
               'influence_froid', 'influence_chaud',
               'froid_diffusion_verticale', 'chaud_diffusion_verticale',
               'dT_dt', 'temp_lag1', 'dist_center_x_t']

resultats_separe_detail = []          # 1 ligne par (modèle, phase)
predictions_separe = {nom: {'y_test': [], 'y_pred': []} for nom in top3_models}

if t_bascule is None:
    print('Aucune bascule détectée à l\'étape 7 -> section 14 ignorée.')
else:
    df_transitoire  = assurer_colonnes(df_transitoire,  df, ['direction_encode'])
    df_stationnaire = assurer_colonnes(df_stationnaire, df, ['direction_encode'])

    for df_subset, nom_phase in [(df_transitoire, 'Transitoire'), (df_stationnaire, 'Stationnaire')]:
        print(f'\n{"="*60}\n{nom_phase.upper()}  ({len(df_subset):,} mesures brutes)\n{"="*60}')

        dfl = construire_features_phase(
            df_subset, capteur_positions, temp_cols,
            SOURCES_FROID, SOURCES_CHAUD, LARGEUR, LONGUEUR, HAUTEUR, ESPACEMENT_Z,
        )
        if len(dfl) < 30:
            print(f'  Trop peu de données ({len(dfl)} lignes) -> phase ignorée.')
            continue

        X_p = dfl[fcols_phase].values
        y_p = dfl['temperature'].values
        X_p_scaled = StandardScaler().fit_transform(X_p)

        t_p = dfl['t_sec'].values
        order_p = np.argsort(t_p, kind='stable')
        n_p = len(t_p)
        split_p = min(max(int(n_p * 0.8), 1), n_p - 1)
        tr_idx, te_idx = order_p[:split_p], order_p[split_p:]
        X_tr, X_te = X_p_scaled[tr_idx], X_p_scaled[te_idx]
        y_tr, y_te = y_p[tr_idx], y_p[te_idx]

        print(f'  n_long = {len(dfl):,} | Train = {len(X_tr):,} | Test = {len(X_te):,}')

        for nom, m_template in top3_models.items():
            # nouvelle instance à chaque phase (ne pas réutiliser un modèle déjà entraîné)
            m = type(m_template)(**m_template.get_params())
            m.fit(X_tr, y_tr)
            pred = m.predict(X_te)
            rmse = np.sqrt(mean_squared_error(y_te, pred))
            mae  = mean_absolute_error(y_te, pred)
            r2   = r2_score(y_te, pred)
            resultats_separe_detail.append({
                'Modèle': nom, 'Phase': nom_phase, 'n_test': len(y_te),
                'RMSE (°C)': rmse, 'MAE (°C)': mae, 'R2': r2,
            })
            predictions_separe[nom]['y_test'].append(y_te)
            predictions_separe[nom]['y_pred'].append(pred)
            print(f'    {nom:14s} RMSE={rmse:.4f}°C  MAE={mae:.4f}°C  R²={r2:.4f}')

tableau_separe_detail = pd.DataFrame(resultats_separe_detail).round(4)
print('\nDétail par phase :')
display(tableau_separe_detail)


## 15 — Comparaison finale (6 cas)

Pour chaque modèle, la ligne **Séparé** regroupe les prédictions des deux phases
(transitoire + stationnaire) et recalcule RMSE/MAE/R² sur l'ensemble regroupé —
un score global comparable à la ligne **Principal**, plutôt qu'une simple moyenne
des deux RMSE (qui ne serait pas mathématiquement correcte).

In [ ]:
resultats_finaux = list(resultats_principal)  # 3 lignes déjà calculées (section 13)

for nom in top3_models:
    y_te_all = predictions_separe[nom]['y_test']
    y_pr_all = predictions_separe[nom]['y_pred']
    if not y_te_all:
        continue
    y_te_concat = np.concatenate(y_te_all)
    y_pr_concat = np.concatenate(y_pr_all)
    rmse = np.sqrt(mean_squared_error(y_te_concat, y_pr_concat))
    mae  = mean_absolute_error(y_te_concat, y_pr_concat)
    r2   = r2_score(y_te_concat, y_pr_concat)
    resultats_finaux.append({'Modèle': nom, 'Approche': 'Séparé',
                              'RMSE (°C)': rmse, 'MAE (°C)': mae, 'R2': r2,
                              'n_test': len(y_te_concat)})

tableau_final = pd.DataFrame(resultats_finaux).round(4)
tableau_final = tableau_final.sort_values(['Modèle', 'Approche']).reset_index(drop=True)
print(f'Comparaison finale — {len(tableau_final)} cas ({len(top3_models)} modèles × 2 approches) :')
display(tableau_final)

# ── Graphique comparatif RMSE ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
noms_modeles = list(top3_models.keys())
x = np.arange(len(noms_modeles))
largeur_barre = 0.35

rmse_principal = [tableau_final[(tableau_final['Modèle'] == m) & (tableau_final['Approche'] == 'Principal')]['RMSE (°C)'].iloc[0] for m in noms_modeles]
rmse_separe    = [tableau_final[(tableau_final['Modèle'] == m) & (tableau_final['Approche'] == 'Séparé')]['RMSE (°C)'].iloc[0] for m in noms_modeles]

ax.bar(x - largeur_barre/2, rmse_principal, largeur_barre, label='Principal', color='steelblue')
ax.bar(x + largeur_barre/2, rmse_separe,    largeur_barre, label='Séparé',    color='seagreen')
ax.set_xticks(x); ax.set_xticklabels(noms_modeles)
ax.set_ylabel('RMSE (°C)'); ax.set_title('RMSE — Principal vs Séparé, par modèle')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

print(
    '\nInterprétation : si \'Séparé\' a un RMSE plus bas que \'Principal\' pour un '
    'modèle donné, entraîner des modèles distincts par phase (stationnaire/transitoire) '
    'améliore la précision pour ce modèle -- au prix de la complexité de devoir '
    'détecter la phase avant de choisir quel modèle utiliser.'
)
